# Training on Google Colab -- `eileen-omni-ui` (Omni UI branch)

This trains the **dual-branch fusion, multi-label** model currently on `eileen-omni-ui`:
raw-IQ branch (dilated 1D-CNN) + STFT branch (2D-CNN), fused via energy-gated
attention pooling, output is a per-class **sigmoid** (not softmax) over all
8 classes (BPSK/QPSK/16QAM/64QAM/LFM_RADAR/FHSS/JAMMING/NOISE_FLOOR) -- so a
window with a jammer overlaid on a real signal correctly reads as BOTH
classes present, instead of forcing one winner.

`configs/default.yaml` already carries a **calibrated per-class threshold for
every class**, not just the 3 judged ones (LFM_RADAR/FHSS/JAMMING) -- civilian
classes and NOISE_FLOOR were the last two groups added, both via
`scripts/calibrate_thresholds.py` against a real checkpoint. Retraining changes
the model's probability calibration, which is why this notebook re-runs that
calibration step after training instead of trusting whatever is already in the
config file.

Colab is a **different computer** -- it cannot see your local drive. So this
notebook:

1. pulls the **code** in from GitHub
2. pulls the **data** in by upload (the small processed arrays, not raw
   RadioML/RadChar -- those are multiple GB and stay on your machine)
3. trains (ensemble + single model), re-calibrates thresholds against the
   fresh checkpoints, checks the >80% judged-class recall gate, and measures
   seed-to-seed variance so a single lucky run isn't mistaken for a real result
4. pushes **everything** back out to your machine

## Read this before you start

**Colab's disk is temporary.** Everything under `/content/` is deleted when
the session ends -- idle timeout, or ~12 hours maximum. If you train for 20
minutes and close the tab without running the download cell, the model is
gone.

**Enable the GPU first:** Runtime -> Change runtime type -> Hardware
accelerator -> GPU. Do this *before* running anything; switching later
restarts the session and wipes your uploads.

**Run cells top to bottom, in order.** Sanity-check runs *before* any
training, and the dataset is never rebuilt inside Colab -- it is built
locally (where RadioML/RadChar live) and uploaded as three small arrays.

**"High accuracy" here means clearing the 80% recall bar on LFM_RADAR/FHSS/
JAMMING with real margin, confirmed by the ensemble + variance run, not a
single lucky number.** Civilian-class recall and JAMMING's seed-to-seed
variance are documented, known limitations of this pipeline (see
`configs/default.yaml`'s comments) -- this notebook does not make those
disappear, it measures where the model actually stands.

## 1. Get the code

**The branch must be pushed to GitHub first** -- Colab clones from GitHub,
it cannot see a branch that only exists on your local machine. Check with
`git status` locally; if it says anything other than
"up to date with 'origin/eileen-omni-ui'", push first:
`git push -u origin eileen-omni-ui`.

If the repo is private, cloning fails. Either make it public, or upload a
zip of the repo instead.

In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -b eileen-omni-ui https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!pwd

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, matplotlib.
# Only these are missing (gradio is NOT needed here -- that's the local
# console, src/ui/, not the training pipeline):
!pip install -q pyyaml h5py

## 2. Check the GPU is actually attached

If this says `CUDA: False`, you skipped the Runtime -> Change runtime type
step. Training still works on CPU, just much slower.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. Get the data in

Upload the three arrays from your machine:

    data/processed/X.npy
    data/processed/y.npy
    data/processed/snr_labels.npy

Build these **locally, on this branch, with RadioML/RadChar available**:

    python -m src.data.build_dataset

**Check `data/processed/X.npy`'s shape locally before uploading.**
`configs/default.yaml` is sized for roughly 58,800 total examples once
composites/mixtures are included -- if your local `data/processed/` only has
a few hundred examples, it's a stale/partial build (e.g. left over from a
quick UI/scenario test), not the real dataset, and training on it will not
produce a trustworthy number. Rebuild with the command above first.

**Shape contract**: `y.npy` is `(N, 8)` -- multi-hot, one column per class,
not `(N,)` with one integer class index per row. The sanity check below
fails loudly rather than silently training on the wrong label format if an
old single-label `y.npy` gets uploaded by mistake.

**Use this cell OR the Drive cell below, not both.**

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('data/processed', exist_ok=True)
print('Select X.npy, y.npy and snr_labels.npy (you can pick all three at once)')
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'data/processed/{name}')
!ls -la data/processed/

### Alternative: mount Google Drive

Better if you will run this repeatedly -- the files persist between
sessions, so you upload once instead of every time. Put the arrays in a
`sedic/` folder in your Drive first.

**Commented out on purpose** -- this is an alternative to the upload cell
above, not an extra step. Uncomment and run *instead of* the upload cell,
not after it.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/processed
# !cp /content/drive/MyDrive/sedic/*.npy data/processed/

## 4. Sanity check -- BEFORE spending any GPU time

Confirms the code runs and the data loaded in the right (multi-label) shape,
and shows the actual class balance you're about to train on. Takes seconds.
**If the pytest run or the shape assertion fails, stop -- do not run the
training cells below**; fix the data/code first.

In [ ]:
!python -m pytest -q

import numpy as np
from src.config import CLASSES

X = np.load('data/processed/X.npy')
y = np.load('data/processed/y.npy')
print('X:', X.shape, X.dtype)
print('y:', y.shape, y.dtype, f'(expect (N, {len(CLASSES)}) -- multi-hot, NOT (N,))')
assert y.ndim == 2 and y.shape[1] == len(CLASSES), (
    'y.npy looks like the OLD single-label format, or CLASSES has changed -- '
    'rebuild the dataset on this branch (python -m src.data.build_dataset) '
    'and re-upload.'
)
if len(X) < 10000:
    print(f'WARNING: only {len(X)} examples -- this looks like a partial/stale '
          f'build, not the full ~58,800-example dataset. Recommended: rebuild '
          f'locally before trusting anything below.')

n_composite = int((y.sum(axis=1) > 1).sum())
print(f'composite (>1 class present) windows: {n_composite} / {len(y)}')
for i, c in enumerate(CLASSES):
    print(f'  {c:<12} {int(y[:, i].sum()):>6}  present')

## 5. Train

1. **Ensemble** (`train_ensemble.py`) -- 5 seeds, sigmoid outputs averaged.
   This is the main result: seed variance on this project has been measured
   up to ~10 points on JAMMING recall alone, so a single run cannot reliably
   answer "is it 80%" on its own.
2. **Variance measurement** -- quantifies that swing directly. Prints only,
   no file is written, so **copy the printed spread numbers somewhere before
   you move on**.
3. **Re-calibrate thresholds against THESE checkpoints** -- every class's
   `configs/default.yaml` threshold was picked against a *previous* model.
   Retraining shifts the model's probability calibration (how confident it
   is, not just how accurate), so the old thresholds may no longer be the
   right cut points for the new checkpoints. This step sweeps fresh
   candidates on the validation split of the data you just trained on and
   prints a ready-to-paste block -- it does **not** edit any file for you.
4. **Single model** (`src.train`) -- a plain baseline run, useful to quote
   alongside the ensemble result in the brief.
5. **Evaluate both** -- the single model against `evals/scorecard.json`, and
   the ensemble (`--ensemble`) for the full 8-class report (confusion
   matrix, accuracy-vs-SNR, comms-vs-jamming) on what would actually be
   submitted, not just the judged-class numbers `train_ensemble.py` prints.

Watch `val_loss`/`val_bit_acc` in the single-model run: falling loss means
learning; `val_bit_acc` is per-class-bit accuracy now, not "did we guess the
one right class" (multiple classes can be true on the same window).

In [ ]:
!python scripts/train_ensemble.py --models 5

In [ ]:
!python scripts/measure_variance.py --runs 5

### Re-calibrate thresholds against the fresh ensemble

Read the printed table, then manually update `multilabel_thresholds_per_class`
in `configs/default.yaml` **locally** (not in this Colab session -- it gets
deleted) before your next `evaluate.py`/`infer.py` run anywhere. The script
deliberately does not touch any file for you -- see its module docstring for
why (val/test leakage this replaced).

In [ ]:
!python scripts/calibrate_thresholds.py --ensemble --n-models 5

In [ ]:
!python -m src.train

In [ ]:
!python -m src.evaluate

In [ ]:
# Full 8-class report (confusion matrix, accuracy-vs-SNR, comms-vs-jamming,
# coarse tiers) for the ensemble actually being submitted -- not just the
# judged-class recall/precision train_ensemble.py's own scorecard reports.
!python -m src.evaluate --ensemble --n-models 5

### Is it 80%? -- the headline answer

Reads back both scorecards and states PASS/FAIL plainly, so you do not have
to scan the printed logs above. `evals/scorecard.json` gets overwritten by
whichever `evaluate.py` call ran last (single vs `--ensemble`) -- this reads
it right after the ensemble evaluate cell above, so it reflects the ensemble.

In [ ]:
import json

print('--- Ensemble, full report (src/evaluate.py --ensemble) ---')
sc = json.load(open('evals/scorecard.json'))
for cls, r in sc['benchmark']['judged_classes'].items():
    print(f"  {cls:<12} recall={r['recall']:.4f}  {'PASS' if r['passed'] else 'FAIL'}")
print(f"  OVERALL: {'PASS' if sc['benchmark']['passed'] else 'FAIL'}")
print()
print('Empty-channel (NOISE_FLOOR) recall:', 
      sc['per_class']['NOISE_FLOOR']['recall'],
      '| precision:', sc['per_class']['NOISE_FLOOR']['precision'])
print('Coarse tier recall:', sc['coarse_tier']['per_tier_recall'])

print()
print('--- Ensemble, judged-class-only scorecard (scripts/train_ensemble.py) ---')
ens = json.load(open('evals/ensemble_scorecard.json'))
for cls, recall in ens['ensemble'].items():
    passed = recall >= 0.80
    print(f"  {cls:<12} recall={recall:.4f}  {'PASS' if passed else 'FAIL'}")
print(f"  OVERALL: {'PASS' if ens['passed'] else 'FAIL'}")
print()
print('The ensemble number, AFTER re-calibrating thresholds against it, is the '
      'one to trust -- it cancels seed-initialisation noise (see the variance '
      'cell above) and uses thresholds actually matched to these checkpoints.')

### Preview the plots before downloading

`diagnose_jamming.py`/`diagnose_radar.py` (sub-type breakdowns) are **not
included here** -- they still assume the old single-label `argmax` output
and have not been updated for multi-label yet. Diagnostic-only (not part of
the scored benchmark), safe to skip.

In [ ]:
from IPython.display import Image, display
display(Image('evals/confusion_matrix.png'))
display(Image('evals/accuracy_vs_snr.png'))

## 6. Get everything out -- DO NOT SKIP THIS

This is the step people forget. Everything above is deleted when the
session ends. Save the downloads into `results/` and `evals/` in your local
repo, and paste the calibrated thresholds from the cell above into
`configs/default.yaml` before you commit anything.

In [ ]:
from google.colab import files
import os, glob

paths = [
    'results/best_model.pt',
    'evals/scorecard.json',
    'evals/confusion_matrix.png',
    'evals/accuracy_vs_snr.png',
    'evals/ensemble_scorecard.json',
] + sorted(glob.glob('results/ensemble_*.pt'))

for path in paths:
    if os.path.exists(path):
        files.download(path)
    else:
        print('missing:', path)

### Or save straight to Drive (survives the session)

In [ ]:
# !mkdir -p /content/drive/MyDrive/sedic/run-YYYY-MM-DD
# !cp results/best_model.pt results/ensemble_*.pt evals/*.json evals/*.png /content/drive/MyDrive/sedic/run-YYYY-MM-DD/